# Phase 2 — Temporal Split + Popularity Baseline
Input: data/processed/interactions.parquet (from Phase 1)

## Cell 1: Compute split dates
Split by timestamp percentile (80th / 90th), not row index — train on everything before T1, validate on [T1, T2), test on [T2, end]. Save the three splits for reuse by later phases.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

interactions = pd.read_parquet(PROCESSED_DIR / "interactions.parquet")
print("loaded interactions:", interactions.shape)

T1 = interactions["timestamp"].quantile(0.80)
T2 = interactions["timestamp"].quantile(0.90)

train = interactions[interactions["timestamp"] < T1].reset_index(drop=True)
val = interactions[(interactions["timestamp"] >= T1) & (interactions["timestamp"] < T2)].reset_index(drop=True)
test = interactions[interactions["timestamp"] >= T2].reset_index(drop=True)

print("T1 (train/val boundary):", pd.to_datetime(T1, unit="ms"))
print("T2 (val/test boundary): ", pd.to_datetime(T2, unit="ms"))
print()
print(f"train: {len(train):>9,} events  ({len(train)/len(interactions):5.1%})")
print(f"val:   {len(val):>9,} events  ({len(val)/len(interactions):5.1%})")
print(f"test:  {len(test):>9,} events  ({len(test)/len(interactions):5.1%})")
print()
print("train date range:", train['datetime'].min(), "->", train['datetime'].max())
print("val date range:  ", val['datetime'].min(), "->", val['datetime'].max())
print("test date range: ", test['datetime'].min(), "->", test['datetime'].max())

train.to_parquet(PROCESSED_DIR / "train.parquet", index=False)
val.to_parquet(PROCESSED_DIR / "val.parquet", index=False)
test.to_parquet(PROCESSED_DIR / "test.parquet", index=False)

with open(PROCESSED_DIR / "split_dates.json", "w") as f:
    json.dump({"T1_ms": float(T1), "T2_ms": float(T2)}, f, indent=2)
print("\nsaved train.parquet, val.parquet, test.parquet, split_dates.json")

loaded interactions: (2755641, 6)


T1 (train/val boundary): 2015-08-18 04:23:20.445000
T2 (val/test boundary):  2015-09-02 17:49:14.032000

train: 2,204,512 events  (80.0%)
val:     275,564 events  (10.0%)
test:    275,565 events  (10.0%)

train date range: 2015-05-03 03:00:04.384000 -> 2015-08-18 04:23:01.129000
val date range:   2015-08-18 04:23:20.445000 -> 2015-09-02 17:49:11.563000
test date range:  2015-09-02 17:49:14.032000 -> 2015-09-18 02:59:47.788000



saved train.parquet, val.parquet, test.parquet, split_dates.json


## Cell 2: Leakage check
Assert the split boundaries are strictly ordered, and that every user's train-side events precede T1.

In [2]:
# Boundary check: no chronological overlap between splits
assert train["timestamp"].max() < val["timestamp"].min(), "train/val boundary violated"
assert val["timestamp"].max() < test["timestamp"].min(), "val/test boundary violated"
print("boundary check: train.max < val.min < val.max < test.min -> PASSED")

# Per-user check: every train-side event for every user must precede T1 by construction.
# Verify explicitly rather than trusting the filter.
violations = (train.groupby("visitorid")["timestamp"].max() >= T1).sum()
print(f"\nusers with a train event >= T1: {violations} (expect 0)")
assert violations == 0, "leakage check FAILED"

users_in_val = val["visitorid"].nunique()
warm_frac = val["visitorid"].isin(train["visitorid"]).mean()
print(f"\nval users: {users_in_val:,}")
print(f"val events from users also seen in train (warm): {warm_frac:.1%}")

print("\nleakage check PASSED")

boundary check: train.max < val.min < val.max < test.min -> PASSED



users with a train event >= T1: 0 (expect 0)

val users: 156,954
val events from users also seen in train (warm): 16.5%

leakage check PASSED


## Cell 3: Popularity baseline
`popularity_score(item) = transactions_train*4 + addtocarts_train*2 + views_train*1`, computed on train only.

Category-aware variant: for each user, use their train-derived top category to rank train-only items within that category; fall back to global top-K for cold-start users or thin categories. The `categoryid` property is time-varying, so it is re-derived here from the raw `item_properties` files using only records with `timestamp < T1` — reusing a Phase-1 snapshot would leak post-split category changes.

In [3]:
RAW_DIR = Path("../data/raw")
EVENT_WEIGHTS = {"view": 1, "addtocart": 2, "transaction": 4}

# --- Global popularity score (train only) ---
popularity_score = (
    train.assign(w=train["event"].map(EVENT_WEIGHTS))
    .groupby("itemid")["w"].sum()
    .sort_values(ascending=False)
)
print("popularity_score: computed for", len(popularity_score), "items (train-only)")
print("\ntop 10 items by popularity:\n", popularity_score.head(10))

# --- categoryid, re-derived leakage-safely: only records timestamp < T1 ---
dtypes = {"itemid": "int32", "property": "category", "value": "object"}
cat_parts = []
for fname in ["item_properties_part1.csv", "item_properties_part2.csv"]:
    df = pd.read_csv(RAW_DIR / fname, dtype=dtypes)
    df = df[(df["property"] == "categoryid") & (df["timestamp"] < T1)]
    cat_parts.append(df[["itemid", "timestamp", "value"]])
cat_raw = pd.concat(cat_parts, ignore_index=True)
item_category = (
    cat_raw.sort_values("timestamp").drop_duplicates("itemid", keep="last")
    .set_index("itemid")["value"]
)
print(f"\nitem_category (pre-T1 snapshot): {len(item_category):,} items mapped to a category")

# --- Each user's top category, from train interactions only ---
train_cat = train.assign(categoryid=train["itemid"].map(item_category)).dropna(subset=["categoryid"])
user_top_category = (
    train_cat.groupby("visitorid")["categoryid"]
    .agg(lambda s: s.value_counts().idxmax())
)
print(f"user_top_category: derived for {len(user_top_category):,} users "
      f"({len(user_top_category) / train['visitorid'].nunique():.1%} of train users)")

popularity_score: computed for 212915 items (train-only)

top 10 items by popularity:
 itemid
461686    2509
5411      2157
187946    1830
257040    1783
309778    1709
370653    1625
7943      1455
298009    1453
369447    1377
48030     1309
Name: w, dtype: int64



item_category (pre-T1 snapshot): 411,807 items mapped to a category


user_top_category: derived for 984,801 users (87.6% of train users)


## Cell 4: Generate baseline recommendations
For each user in val: category-aware top-K if they have a train-derived top category, backfilled with global top-K; pure global top-K for cold-start users. Exclude items the user already purchased in train.

In [4]:
K = 10
BUFFER = 50  # extra candidates pulled before filtering seen purchases, so slicing to K is still safe

global_topk = popularity_score.index.tolist()

# Precompute per-category item ranking (train-only), for the category-aware variant
item_cat_pop = pd.DataFrame({"itemid": popularity_score.index, "score": popularity_score.values})
item_cat_pop["categoryid"] = item_cat_pop["itemid"].map(item_category)
item_cat_pop = item_cat_pop.dropna(subset=["categoryid"])
category_topk = (
    item_cat_pop.sort_values("score", ascending=False)
    .groupby("categoryid")["itemid"]
    .apply(list)
    .to_dict()
)
print(f"category_topk built for {len(category_topk):,} categories")

# Purchases already made in train, to exclude from recommendations
purchased_in_train = (
    train[train["event"] == "transaction"].groupby("visitorid")["itemid"].apply(set).to_dict()
)
print(f"users with a train purchase to exclude: {len(purchased_in_train):,}")

def recommend_for_user(uid, k=K):
    excluded = purchased_in_train.get(uid, set())
    cat = user_top_category.get(uid)
    if cat is not None and cat in category_topk:
        candidates = [i for i in category_topk[cat][: k + BUFFER] if i not in excluded][:k]
        if len(candidates) < k:
            fill = [i for i in global_topk[: k + BUFFER] if i not in excluded and i not in candidates]
            candidates += fill[: k - len(candidates)]
    else:
        candidates = [i for i in global_topk[: k + BUFFER] if i not in excluded][:k]
    return candidates

val_users = val["visitorid"].unique()
recs_rows = []
for uid in val_users:
    recs = recommend_for_user(uid)
    for rank, itemid in enumerate(recs, start=1):
        recs_rows.append((uid, rank, itemid))

baseline_recs = pd.DataFrame(recs_rows, columns=["visitorid", "rank", "itemid"])
baseline_recs.to_parquet(PROCESSED_DIR / "baseline_recs_val.parquet", index=False)

print(f"\ngenerated recs for {baseline_recs['visitorid'].nunique():,} / {len(val_users):,} val users")
print("mean recs per user:", baseline_recs.groupby("visitorid").size().mean())
print("\nsample recommendations for 3 users:")
for uid in val_users[:3]:
    cat = user_top_category.get(uid)
    print(f"  user {uid} (top_category={cat}): {recommend_for_user(uid)}")

category_topk built for 1,080 categories


users with a train purchase to exclude: 9,343



generated recs for 156,954 / 156,954 val users
mean recs per user: 10.0

sample recommendations for 3 users:
  user 782887 (top_category=None): [461686, 5411, 187946, 257040, 309778, 370653, 7943, 298009, 369447, 48030]
  user 269420 (top_category=242): [144704, 303229, 321712, 338148, 130352, 318785, 322513, 45589, 29080, 9376]
  user 475017 (top_category=None): [461686, 5411, 187946, 257040, 309778, 370653, 7943, 298009, 369447, 48030]


## Cell 5: Evaluate baseline
Compute Recall@10, NDCG@10, Coverage@10, AddToCart-Recall@10, Purchase-Recall@10 against the val set.

Relevance definitions:
- **Recall@10 / NDCG@10**: relevant = any item the user interacted with (view/cart/purchase) during val.
- **AddToCart-Recall@10**: relevant = items the user added to cart during val.
- **Purchase-Recall@10**: relevant = items the user purchased during val.
- **Coverage@10**: unique items recommended across all users / catalog size (items seen in train).

Only users who have at least one relevant item are included in each metric's average (standard practice — recall/NDCG are undefined for users with zero relevant items).

In [5]:
import math

recs_by_user = (
    baseline_recs.sort_values(["visitorid", "rank"])
    .groupby("visitorid")["itemid"].apply(list)
    .to_dict()
)

relevant_any = val.groupby("visitorid")["itemid"].apply(set).to_dict()
relevant_cart = val[val["event"] == "addtocart"].groupby("visitorid")["itemid"].apply(set).to_dict()
relevant_purchase = val[val["event"] == "transaction"].groupby("visitorid")["itemid"].apply(set).to_dict()

def recall_at_k(recs_by_user, relevant):
    scores = []
    for uid, rel in relevant.items():
        if not rel:
            continue
        rec = recs_by_user.get(uid, [])
        hit = len(set(rec) & rel)
        scores.append(hit / len(rel))
    return float(np.mean(scores)) if scores else 0.0, len(scores)

def ndcg_at_k(recs_by_user, relevant, k=K):
    scores = []
    for uid, rel in relevant.items():
        if not rel:
            continue
        rec = recs_by_user.get(uid, [])
        dcg = sum(1.0 / math.log2(i + 2) for i, item in enumerate(rec[:k]) if item in rel)
        idcg = sum(1.0 / math.log2(i + 2) for i in range(min(k, len(rel))))
        scores.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(scores)) if scores else 0.0, len(scores)

def coverage_at_k(recs_by_user, catalog_size):
    unique_recommended = set()
    for rec in recs_by_user.values():
        unique_recommended.update(rec)
    return len(unique_recommended) / catalog_size, len(unique_recommended)

catalog_size = train["itemid"].nunique()

recall, n_recall = recall_at_k(recs_by_user, relevant_any)
ndcg, n_ndcg = ndcg_at_k(recs_by_user, relevant_any)
coverage, n_unique = coverage_at_k(recs_by_user, catalog_size)
addtocart_recall, n_cart = recall_at_k(recs_by_user, relevant_cart)
purchase_recall, n_purchase = recall_at_k(recs_by_user, relevant_purchase)

metrics = {
    "recall_at_10": recall,
    "ndcg_at_10": ndcg,
    "coverage_at_10": coverage,
    "addtocart_recall_at_10": addtocart_recall,
    "purchase_recall_at_10": purchase_recall,
}

print(f"Recall@10:           {recall:.4f}   (n_users={n_recall:,})")
print(f"NDCG@10:             {ndcg:.4f}   (n_users={n_ndcg:,})")
print(f"Coverage@10:         {coverage:.4f}   ({n_unique:,} / {catalog_size:,} catalog items)")
print(f"AddToCart-Recall@10: {addtocart_recall:.4f}   (n_users={n_cart:,})")
print(f"Purchase-Recall@10:  {purchase_recall:.4f}   (n_users={n_purchase:,})")

with open(PROCESSED_DIR / "baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nsaved data/processed/baseline_metrics.json")

Recall@10:           0.0143   (n_users=156,954)
NDCG@10:             0.0083   (n_users=156,954)
Coverage@10:         0.0360   (7,658 / 212,915 catalog items)
AddToCart-Recall@10: 0.0194   (n_users=4,235)
Purchase-Recall@10:  0.0251   (n_users=1,333)

saved data/processed/baseline_metrics.json
